 # CM3010 Midterm Project Report — Austin Animal Center Outcomes



# Abstract

This project investigates the transformation of a flat, real-world dataset into a structured relational database. Utilizing open data from the Austin Animal Center, which contains over 170,000 records documenting various animal outcomes such as adoptions and transfers, the project emphasizes data modeling, normalization, and structured query capabilities. The primary objective is to redesign the dataset to support relational database principles and SQL-based access, establishing a solid foundation for future integration with web-based applications. A systematic process is employed, including dataset assessment, entity-relationship modeling, normalization, relational database implementation using MySQL, and the development of queries aligned with specific research questions.


# Introduction

The Austin Animal Center Outcomes dataset, made publicly available by the City of Austin, serves as a significant resource for examining patterns within one of the largest no-kill animal shelters in the United States. The dataset records essential attributes including animal identification, species, breed, color, age, sex, type of outcome, and date of outcome.

The intent of this project is to evaluate and restructure the dataset using normalization techniques to create an efficient and coherent relational schema. Through the application of relational database design principles and structured query language (SQL), the project facilitates in-depth exploration of animal welfare trends, such as adoption frequencies, breed-specific outcomes, and seasonal variances in shelter activity.

This report outlines the methodological stages undertaken in the project: initial dataset evaluation, development of an entity-relationship model, normalization to the third normal form, implementation of the schema in MySQL, and execution of analytical queries addressing pre-defined research objectives. This structured approach not only highlights the practical utility of relational database theory but also underscores its application within the domain of animal welfare data management.

# Data Cleaning and Preparation


### Step 1: Import Libraries and Load the Dataset

We load the CSV file using pandas and examine the first few rows and structure of the dataset.

In [9]:
import pandas as pd

df = pd.read_csv("Austin_Animal_Center_Outcomes__10_01_2013_to_05_05_2025__20250606.csv")
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Animal ID         173775 non-null  object
 1   Date of Birth     173775 non-null  object
 2   Name              123991 non-null  object
 3   DateTime          173775 non-null  object
 4   MonthYear         173775 non-null  object
 5   Outcome Type      173729 non-null  object
 6   Outcome Subtype   79660 non-null   object
 7   Animal Type       173775 non-null  object
 8   Sex upon Outcome  173774 non-null  object
 9   Age upon Outcome  173766 non-null  object
 10  Breed             173775 non-null  object
 11  Color             173775 non-null  object
dtypes: object(12)
memory usage: 15.9+ MB


### Step 2: Clean and Standardize Column Names

We convert all column names to lowercase and replace spaces with underscores for consistency and ease of use in SQL and code.


In [2]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df.columns

Index(['animal_id', 'date_of_birth', 'name', 'datetime', 'monthyear',
       'outcome_type', 'outcome_subtype', 'animal_type', 'sex_upon_outcome',
       'age_upon_outcome', 'breed', 'color'],
      dtype='object')

### Step 3: Drop Rows with Missing Essential Data

We remove rows that are missing critical fields like `animal_id` or `outcome_type`.


In [3]:
df = df.dropna(subset=['animal_id', 'outcome_type'])

# Confirm drop was successful
df[['animal_id', 'outcome_type']].isnull().sum()


animal_id       0
outcome_type    0
dtype: int64

### Step 4: Fill Non-Essential Missing Data

We replace missing `name`, `sex_upon_outcome`, and `age_upon_outcome` with "Unknown" to avoid nulls in MySQL.


In [4]:
df['name'] = df['name'].fillna('Unknown')
df['sex_upon_outcome'] = df['sex_upon_outcome'].fillna('Unknown')
df['age_upon_outcome'] = df['age_upon_outcome'].fillna('Unknown')

df[['name', 'sex_upon_outcome', 'age_upon_outcome']].isnull().sum()

name                0
sex_upon_outcome    0
age_upon_outcome    0
dtype: int64

### Step 5: Convert Date Columns to Datetime Format

We convert the `datetime` and `date_of_birth` columns from strings to actual datetime objects so we can calculate durations later (like animal age at outcome). We’ll also display the column types to verify the conversion worked.


In [5]:
# Convert date columns to datetime format
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df = df.dropna(subset=['datetime', 'date_of_birth'])

# Confirm the data types of these columns
df[['datetime', 'date_of_birth']].dtypes


datetime         datetime64[ns, UTC-05:00]
date_of_birth               datetime64[ns]
dtype: object

### Step 6: Create a Derived Column for Age in Days

Now that both `datetime` and `date_of_birth` are properly formatted, we can subtract them to calculate the animal’s age (in days) at the time of their outcome. This new column can help answer questions about age-based trends.


In [6]:
# Ensure both datetime columns are tz-naive
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce').dt.tz_localize(None)
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce').dt.tz_localize(None)

df['age_days'] = (df['datetime'] - df['date_of_birth']).dt.days

# Show a preview of the result
df[['animal_id', 'datetime', 'date_of_birth', 'age_days']].head()


,animal_id,datetime,date_of_birth,age_days
0,A668305,2013-12-02,2012-12-01,366
1,A673335,2014-02-22,2012-02-22,731
2,A675999,2014-04-07,2013-04-03,369
4,A680855,2014-06-10,2014-05-25,16
5,A680857,2014-06-10,2014-05-25,16


### Step 7: Preview Unique Values for Normalization

To design our relational database properly, we need to identify which columns contain repeated values that could be normalized into separate tables (e.g., `breeds`, `colors`, `outcome_types`, etc.).


In [7]:
# Preview number of unique values in key categorical columns
print("Unique animal types:", df['animal_type'].unique())
print("\nUnique outcome types:", df['outcome_type'].unique())
print("\nUnique outcome subtypes:", df['outcome_subtype'].dropna().unique())
print("\nUnique breeds:", df['breed'].nunique())
print("\nTop 10 most common breeds:")
print(df['breed'].value_counts().head(10))
print("\nUnique colors:", df['color'].nunique())


Unique animal types: ['Other' 'Bird' 'Dog' 'Cat' 'Livestock']

Unique outcome types: ['Transfer' 'Euthanasia' 'Adoption' 'Return to Owner' 'Died' 'Missing'
 'Disposal' 'Relocate' 'Rto-Adopt']

Unique outcome subtypes: ['Partner' 'Suffering' 'Foster' 'Offsite' 'In Kennel' 'Enroute'
 'In Foster' 'Aggressive' 'SCRP' 'Rabies Risk' 'At Vet' 'Possible Theft'
 'Medical' 'Snr' 'Field' 'Customer S' 'In Surgery' 'Out State' 'Emergency']

Unique breeds: 407

Top 10 most common breeds:
breed
Domestic Shorthair Mix       844
Domestic Shorthair           556
Pit Bull Mix                 221
Labrador Retriever Mix       195
Chihuahua Shorthair Mix      184
Domestic Medium Hair Mix      93
German Shepherd Mix           58
Domestic Medium Hair          45
Australian Cattle Dog Mix     45
Boxer Mix                     39
Name: count, dtype: int64

Unique colors: 183


### Step 8: Save Cleaned Sample for MySQL Import

To make testing and database setup easier, we export the first 5000 cleaned rows to a new CSV file. This sample will be used to populate our MySQL tables without overloading the system.


In [8]:
# Select first 5000 rows from the cleaned DataFrame
df_clean = df.head(5000)

# Export to CSV without the index column
df_clean.to_csv('animal_clean_sample.csv', index=False)

# Preview the saved sample
df_clean.head()


,animal_id,date_of_birth,name,datetime,monthyear,outcome_type,outcome_subtype,animal_type,sex_upon_outcome,age_upon_outcome,breed,color,age_days
0,A668305,2012-12-01,Unknown,2013-12-02,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow,366
1,A673335,2012-02-22,Unknown,2014-02-22,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray,731
2,A675999,2013-04-03,Unknown,2014-04-07,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green,369
4,A680855,2014-05-25,Unknown,2014-06-10,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black,16
5,A680857,2014-05-25,Unknown,2014-06-10,06-2014,Transfer,Partner,Bird,Unknown,2 weeks,Duck,Yellow/Black,16


# Stage 1: Find and Critique a Dataset

## 1.1 Dataset Selection
The dataset selected for this study is the Austin Animal Center Outcomes dataset, which provides comprehensive records detailing the entry and exit of animals at the largest no-kill municipal shelter in the United States. Published and maintained by the City of Austin through its Open Data Portal, this dataset represents real-world, unstructured data that has not been pre-configured for relational use. As such, it presents an appropriate foundation for tasks involving normalization and relational schema development.

The dataset contains key variables, including animal ID, type, breed, color, age, sex, intake and outcome details, and relevant timestamps. It is publicly accessible and regularly updated in both CSV and API formats, enhancing its utility for academic and technical purposes.ats.


## 1.2 Dataset Assessment

The Austin Animal Center Outcomes dataset was assessed using the criteria outlined in discussion 1.104: quality, level of detail, documentation, interrelation, use, and discoverability. Additionally, the terms of use were reviewed in accordance with the considerations from discussion 1.206. This section outlines the strengths and limitations of the dataset with respect to these criteria and evaluates its suitability for the development of a relational database and web application.

### 2.1 Quality

The dataset is well-maintained and offers a high degree of consistency. It contains 173,775 rows and 12 columns, with each row representing a unique outcome event for an individual animal. Most core fields, such as animal_id, datetime, animal_type, and outcome_type, are well-populated with very few missing values. However, there are some non-critical fields, such as name, outcome_subtype, or age_upon_outcome, that occasionally contain null or incomplete entries.

- Critical fields like animal_id, datetime, and outcome_type are over 99% complete.
- Missing values are mostly limited to optional fields such as name and outcome_subtype.
- Duplicate values are minimal and mostly correspond to repeated visits by the same animal ID (i.e., valid duplicate keys rather than data errors).

### 2.2 Level of Detail

The dataset provides a high level of detail, capturing multiple aspects of each animal’s interaction with the shelter system. Each entry includes attributes such as:

- animal_type (e.g., Dog, Cat, Other)
- breed (specific breed or mixed)
- color
- sex_upon_outcome
- age_upon_outcome
- date_of_birth and datetime (outcome date)

This level of granularity allows for a wide range of analytical questions to be asked, including comparisons across breeds, outcome trends over time, age-based analysis, and more. The data supports breakdowns by multiple demographic and temporal variables, making it highly suitable for both relational modeling and complex queries.

### 2.3 Documentation

The dataset is hosted on the City of Austin’s official Open Data Portal, which offers essential metadata, including:

- A general description of the dataset and its purpose
- Field names and brief explanations for each column
- Metadata such as update frequency, last modified date, and data publisher

While it lacks a downloadable data dictionary or technical schema, the documentation provided is sufficient for understanding the structure and intent of each field. For example, it is clear that datetime refers to the outcome date/time rather than intake, and that animal_id serves as a unique identifier for each case. The field names are self-explanatory and logically structured, reducing ambiguity for developers and analysts.

### 2.4 Interrelation

The dataset is inherently relational, even though it is presented in a flat format. Several columns are suitable for normalization into separate lookup or reference tables. For instance:

- breed and color contain repeated values across thousands of records and can be stored in separate tables.
- outcome_type and outcome_subtype represent categorical data that can be normalized into linked tables.
- animal_type can be associated with one-to-many relationships to breeds and outcomes.

These relational patterns support transformation into a normalized schema, enhancing storage efficiency, reducing redundancy, and enabling advanced query capabilities through relational joins.

### 2.5 Use

This dataset holds significant potential for practical application. Its primary utility lies in supporting operational analysis and strategic planning in animal welfare services. Potential use cases include:

- Monitoring shelter performance and animal outcome trends
- Identifying at-risk animal groups (e.g., by breed or age)
- Planning outreach or adoption events based on seasonal trends
- Supporting animal welfare policies through data-driven decision-making

From an academic perspective, the dataset is particularly useful for designing real-world applications such as interactive dashboards or transparent adoption tools.

### 2.6 Discoverability

The dataset is easily accessible via the City of Austin Open Data Portal. It is well-indexed, categorized under Health and Community Services, and available in multiple formats such as CSV, Excel, and JSON.

- No user authentication is required for data access
- A public API is available for programmatic use
- The portal’s search capabilities also facilitate discovery of related datasets, including animal intake records

These features ensure that the dataset is accessible and replicable for academic, technical, and civic engagement purposes.

### 2.7 Terms of Use

According to the City of Austin’s Open Data Policy, the dataset is openly licensed and intended for unrestricted public use. The license allows:

- Free usage, redistribution, and modification
- No associated costs or fees
- Voluntary attribution to the City of Austin

There are no legal or technical constraints preventing its inclusion in coursework or public-facing projects, provided appropriate acknowledgment of the data source is maintained.

## 1.3 Interest in the Dataset
The Austin Animal Center Outcomes dataset is particularly compelling due to its connection with the day-to-day realities of public animal shelters. It captures the challenges involved in managing, caring for, and rehoming animals. The data provides meaningful insight into how factors such as species, breed, age, and intake type influence animal outcomes. These insights can support real efforts to improve animal welfare, encourage adoptions, and better understand how shelters operate.

From a technical point of view, this dataset is well-suited for building a normalized relational model. Its raw format, size, and structure make it an ideal foundation for applying database design principles. It allows for practical demonstrations of SQL querying and relational design. In addition, it supports the development of web interfaces that allow users to interact with the data in dynamic and informative ways.ion.

## 1.4 Research Questions

To justify a database and SQL approach, the following questions were identified:

1. **What are the most common outcome types for each animal type?**  
   This shows how different species tend to exit the shelter system (e.g., adoption vs. euthanasia).

2. **Which 10 breeds have the highest number of adoptions?**  
   Helps identify breed popularity and potential breed biases.

3. **What is the average time (in days) animals stay in the shelter before an outcome?**  
   Measures efficiency and overall shelter performance.

4. **How does outcome type vary across age groups?**  
   Reveals patterns related to adoptability by life stage.

5. **Which months have the highest number of adoptions?**  
   Supports analysis of seasonal patterns and planning al MySQL database.


# Stage 2: Model Your Data


This section of the report outlines the data modeling process for the Austin Animal Center Outcomes dataset. The process begins with the construction of an Entity-Relationship (E/R) diagram, which is then translated into a relational schema to support normalization and database efficiency. The aim of this stage is to ensure a logically structured and relationally sound design, grounded in best practices from database theory and normalization principles discussed in this module.


### 2.1 Entity-Relationship Model

The E/R model was created to represent the logical structure of the data extracted from the Austin Animal Center Outcomes dataset. It identifies the main entities, their attributes, and the relationships between them. This model helps ensure that the database is normalized and ready for efficient querying.

The five main entities identified are:

- **animals**  
  Attributes: `animal_id`, `name`, `animal_type`, `breed_id`, `color`, `sex_upon_outcome`, `age_upon_outcome`, `date_of_birth`

- **breeds**  
  Attributes: `breed_id`, `breed_name`

- **outcomes**  
  Attributes: `outcome_id`, `animal_id`, `outcome_type_id`, `outcome_subtype_id`, `outcome_date`, `age_days`

- **outcome_types**  
  Attributes: `outcome_type_id`, `outcome_type_name`

- **outcome_subtypes**  
  Attributes: `outcome_subtype_id`, `outcome_subtype_name`


## 2.2 Entity-Relationship Diagram

![Figure 1: Entity-Relationship Diagram](JawLiChing-Database-MidTerms-ERDiagram.png)

*(Figure 1. ER Diagram)*

**Subset Justification:**  
Only outcome data was modeled, as it offers sufficient detail for relational analysis and aligns with the project scope. Intake-related attributes were not used due to missing or incomplete records and were excluded to keep the project focused and feasible.



**Each entity was designed to reduce redundancy and support data consistency. Relationships were identified as follows:**

- Each animal can have one breed → `animals.breed_id → breeds.breed_id`
- Each outcome is linked to one animal → `outcomes.animal_id → animals.animal_id`
- Each outcome has one outcome type and one outcome subtype →  
  `outcomes.outcome_type_id → outcome_types.outcome_type_id`  
  `outcomes.outcome_subtype_id → outcome_subtypes.outcome_subtype_id`


**Cardinalities:**
- One `animal` → many `outcomes`
- One `breed` → many `animals`
- One `outcome_type` → many `outcomes`
- One `outcome_subtype` → many `outcomes`
malization.


## 2.3 Relational Schema

![Figure 1: Relational Schema Diagram](JawLiChing-Database-MidTerms-RelationalSchema.png)

*(Figure 2. Database diagram)*

- **Tables Description**: The schema comprises the following five normalized tables:

  - `animals`: Contains
    - `animal_id` (Primary Key)
    - `name`
    - `animal_type`
    - `breed_id` (Foreign Key → breeds.breed_id)
    - `color`
    - `sex_upon_outcome`
    - `age_upon_outcome`
    - `date_of_birth`

  - `breeds`: Contains
    - `breed_id` (Primary Key)
    - `breed_name`

  - `outcomes`: Contains
    - `outcome_id` (Primary Key)
    - `animal_id` (Foreign Key → animals.animal_id)
    - `outcome_type_id` (Foreign Key → outcome_types.outcome_type_id)
    - `outcome_subtype_id` (Foreign Key → outcome_subtypes.outcome_subtype_id)
    - `outcome_date`
    - `age_days`

  - `outcome_types`: Contains
    - `outcome_type_id` (Primary Key)
    - `outcome_type_name`

  - `outcome_subtypes`: Contains
    - `outcome_subtype_id` (Primary Key)
    - `outcome_subtype_name`


Table 1 below shows all the attributes present in the original dataset. For this project, only a subset of relevant attributes are selected based on their usefulness in answering the research questions and their data completeness. Redundant fields and those with excessive missing values are excluded. Additionally, repeated textual values (e.g., breed, outcome types) are normalized into separate tables to support a relational schema.

| Attributes           | Descriptions                                                                 |
|----------------------|-------------------------------------------------------------------------------|
| animal_id            | Unique identifier for each animal                                            |
| name                 | Name of the animal (if given)                                                |
| animal_type          | Species or general type of animal (e.g., Dog, Cat)                           |
| breed                | Breed or mix of breeds                                                       |
| color                | Color of the animal                                                          |
| sex_upon_outcome     | Sex and neuter/spay status at the time of outcome                            |
| age_upon_outcome     | Age of the animal at the time of outcome (e.g., 2 years, 3 months)           |
| date_of_birth        | Date of birth of the animal                                                  |
| datetime             | Outcome date and time                                                        |
| outcome_type         | General outcome (e.g., Adoption, Euthanasia, Return to Owner)                |
| outcome_subtype      | More specific outcome detail (e.g., Foster, Rabies Risk, Medical)            |
| monthyear            | Month and year of the outcome (used for grouping time-based analysis)        |

*(Table 1. Original dataset attributes)*


### 2.4 Relational Model Normalisation

The database design was carefully evaluated using standard normalization principles. The goal was to remove redundancy, ensure logical data structure, and improve query performance. This design satisfies First Normal Form (1NF), Second Normal Form (2NF), Third Normal Form (3NF), Boyce-Codd Normal Form (BCNF), and Fourth Normal Form (4NF).

#### First Normal Form (1NF)
All columns contain atomic values. There are no repeating groups or arrays. Each record in the `animals` table stores a single breed, a single color, and one outcome per row. This structure ensures clarity and consistency.

#### Second Normal Form (2NF)
Each non-key attribute is fully dependent on the entire primary key. Since most tables use a single-column primary key, partial dependencies are avoided. For example, the breed name is not stored in the `animals` table but instead referenced through a foreign key to the `breeds` table.

#### Third Normal Form (3NF)
There are no transitive dependencies in the schema. Attributes like outcome type and breed name are stored in their respective reference tables. This separation keeps the data clean and prevents duplication.

#### Boyce-Codd Normal Form (BCNF)
Every determinant is a candidate key. For instance, `breed_id` determines `breed_name`, and no non-key fields determine other fields. This removes any risk of unexpected dependencies in the database.

#### Fourth Normal Form (4NF)
There are no multivalued dependencies. Animals with multiple outcomes are stored in individual rows in the `outcomes` table, not as multiple columns. This structure avoids data repetition and allows for flexible expansion in the future.

### Summary of Normalisation

| Normal Form | Satisfied | Explanation |
|-------------|-----------|-------------|
| 1NF         | Yes       | Values are atomic, no repeating fields |
| 2NF         | Yes       | No partial dependencies on composite keys |
| 3NF         | Yes       | No transitive dependencies between non-key fields |
| BCNF        | Yes       | Every determinant is a candidate key |
| 4NF         | Yes       | No multivalued dependencies exist in any table |

The final schema was normalised up to 4NF. This level of structure is suitable for the dataset, which involves multiple one-to-many relationships and repeated categorical values. The result is a stable, scalable, and easy-to-maintain database that supports accurate analysis and reporting.
mes, ensuring a robust and reliable database design.mes, ensuring a robust and reliable database design.mes, ensuring a robust and reliable database design.mes, ensuring a robust and reliable database design.

# Stage 3: Create the Database

This stage involves building the actual database in MySQL using the normalized schema designed in Stage 2. The database structure was created with `CREATE TABLE` commands, a cleaned sample of data was imported, and a set of SQL queries were executed to answer the research questions from Stage 1.


## 3.1 Build the Database Structure in MySQL
To build the database, we used MySQL Workbench and command-line tools in the lab environment. The database was created using the following command:

The schema was then defined using a series of CREATE TABLE statements, each corresponding to a normalized table from the E/R diagram.

The tables were structured to follow third normal form (3NF), ensuring atomic fields, elimination of partial and transitive dependencies, and use of foreign keys to preserve referential integrity. Lookup/reference tables like breeds, outcome_types, and outcome_subtypes were created first to support foreign key relationships.

Each CREATE TABLE command used is documented in the table below.

### Table: SQL Table Creation Commands and Descriptions

| CREATE commands | Descriptions |
|-----------------|--------------|
| `CREATE TABLE breeds ( breed_id INT PRIMARY KEY AUTO_INCREMENT, breed_name VARCHAR(100) );` | Create table called `breeds` with attributes `breed_id` (primary key) and `breed_name`. |
| `CREATE TABLE outcome_types ( outcome_type_id INT PRIMARY KEY AUTO_INCREMENT, outcome_type_name VARCHAR(50) );` | Create table called `outcome_types` with attributes `outcome_type_id` (primary key) and `outcome_type_name`. |
| `CREATE TABLE outcome_subtypes ( outcome_subtype_id INT PRIMARY KEY AUTO_INCREMENT, outcome_subtype_name VARCHAR(50) );` | Create table called `outcome_subtypes` with attributes `outcome_subtype_id` (primary key) and `outcome_subtype_name`. |
| `CREATE TABLE animals ( animal_id VARCHAR(20) PRIMARY KEY, name VARCHAR(100), animal_type VARCHAR(50), breed_id INT, color VARCHAR(50), sex_upon_outcome VARCHAR(50), age_upon_outcome VARCHAR(50), date_of_birth DATE, FOREIGN KEY (breed_id) REFERENCES breeds(breed_id) );` | Create table called `animals` with attributes like `animal_id` (primary key), `name`, `animal_type`, `breed_id` (FK), `color`, `sex_upon_outcome`, `age_upon_outcome`, and `date_of_birth`. |
| `CREATE TABLE outcomes ( outcome_id INT PRIMARY KEY AUTO_INCREMENT, animal_id VARCHAR(20), outcome_type_id INT, outcome_subtype_id INT, outcome_date DATETIME, age_days INT, FOREIGN KEY (animal_id) REFERENCES animals(animal_id), FOREIGN KEY (outcome_type_id) REFERENCES outcome_types(outcome_type_id), FOREIGN KEY (outcome_subtype_id) REFERENCES outcome_subtypes(outcome_subtype_id) );` | Create table called `outcomes` with attributes `outcome_id` (primary key), foreign keys to `animals`, `outcome_types`, and `outcome_subtypes`, along with `outcome_date` and `age_days`. |

*(Table 2. SQL Table Creation Commands and Descriptions)*


## 3.2 Enter Instance Data

To populate the database, a cleaned sample of 5,000 rows was exported from the dataset using Python after preprocessing and filtering invalid or missing values. Once converted into a CSV file named `animal_clean_sample.csv`, a total of 3,583 usable records remained.



### Step 1: Create the staging table (`small_data`)

To simplify data loading, a temporary staging table was created to match the CSV file structure. This allowed the dataset to be imported without requiring immediate joins or foreign key checks.


CREATE TABLE small_data (
  animal_id VARCHAR(20),
  date_of_birth DATE,
  name VARCHAR(100),
  datetime DATETIME,
  monthyear VARCHAR(20),
  outcome_type VARCHAR(50),
  outcome_subtype VARCHAR(50),
  animal_type VARCHAR(50),
  sex_upon_outcome VARCHAR(50),
  age_upon_outcome VARCHAR(50),
  breed VARCHAR(100),
  color VARCHAR(50),
  age_days INT
);

Once the table was ready, the CSV file was moved to the directory specified by MySQL’s secure_file_priv setting. The data was then imported using the following command:



LOAD DATA INFILE 'C:/ProgramData/MySQL/MySQL Server 8.0/Uploads/animal_clean_sample.csv'
INTO TABLE small_data
FIELDS TERMINATED BY ','
ENCLOSED BY '"'
LINES TERMINATED BY '\n'
IGNORE 1 LINES;

To normalize the data, unique values for breeds, outcome types, and subtypes were extracted from the small_data table and inserted into their respective lookup tables:

INSERT IGNORE INTO breeds (breed_name)
SELECT DISTINCT breed FROM small_data;

INSERT IGNORE INTO outcome_types (outcome_type_name)
SELECT DISTINCT outcome_type FROM small_data;

INSERT IGNORE INTO outcome_subtypes (outcome_subtype_name)
SELECT DISTINCT outcome_subtype FROM small_data;

Next, the animals table was populated by joining small_data with the breeds table to resolve the appropriate foreign key for breed_id:

INSERT IGNORE INTO animals
(animal_id, name, animal_type, breed_id, color, sex_upon_outcome, age_upon_outcome, date_of_birth)
SELECT 
  s.animal_id,
  s.name,
  s.animal_type,
  b.breed_id,
  s.color,
  s.sex_upon_outcome,
  s.age_upon_outcome,
  s.date_of_birth
FROM small_data s
LEFT JOIN breeds b ON s.breed = b.breed_name;

Finally, outcome records were inserted into the outcomes table by joining outcome types and subtypes by name to retrieve the correct foreign keys:

INSERT IGNORE INTO outcomes
(animal_id, outcome_type_id, outcome_subtype_id, outcome_date, age_days)
SELECT 
  s.animal_id,
  ot.outcome_type_id,
  ost.outcome_subtype_id,
  s.datetime,
  s.age_days
FROM small_data s
LEFT JOIN outcome_types ot ON s.outcome_type = ot.outcome_type_name
LEFT JOIN outcome_subtypes ost ON s.outcome_subtype = ost.outcome_subtype_name;

A final verification showed that 3,573 unique records were inserted into the animals table and corresponding records were added to the outcomes table. The use of a staging table ensured smooth transformation and accurate mapping of foreign keys across related tables. Once the insertion was complete, the small_data table was no longer needed and could be removed from the schema to maintain cleanliness.

## 3.3 Critical Reflection on the Database Design

The final database design aligns well with the structure and purpose of the project. It supports analysis focused on adoption trends, breed frequency, and outcomes by age group. Through normalization, repeated values like `breeds`, `outcome_types`, and `outcome_subtypes` were placed in separate reference tables. This improves consistency and allows for more flexible querying, supporting the overall goal of uncovering patterns in animal shelter outcomes.

One challenge was handling missing or invalid `datetime` and `date_of_birth` values. These had to be removed during preprocessing to avoid import errors, which slightly reduced the dataset and may have limited time-based analysis. This reflects a common trade-off between data quality and completeness.

Another issue involved duplicate primary keys in the `animals` table. Using `INSERT IGNORE` allowed the import to proceed without interruption, but still produced warnings. These did not impact results but suggest that additional data validation may be needed in future iterations.

The use of a staging table (`small_data`) was a practical solution for importing and transforming raw data before loading it into the normalized structure. This step made foreign key mapping more reliable and ensured cleaner joins, which were important for breed- and outcome-based queries.

Importantly, the design choices align well with the research questions posed in the project. Separating outcomes from animals enables analysis of multiple outcome types per animal and supports questions about which types of outcomes are most common for specific breeds or age groups. The inclusion of outcome_subtypes also allows deeper exploration if more granular outcome categories are added later. 

In summary, the final design balances theory with practical use. It supports the project's goals and enables reliable analysis, despite minor limitations during implementation.

## 3.4 SQL Queries to Answer Research Questions

To evaluate the usefulness of the database and validate its structure, a set of SQL queries were designed to answer the research questions identified in Stage 1. These queries demonstrate how the relational model supports meaningful data analysis.

**Research Question 1. What are the most common outcome types for each animal type?**


This query identifies the most frequent outcome types (such as Transfer or Adoption) for each type of animal (e.g., Dog, Cat, Bird). It does this by joining the animals, outcomes, and outcome_types tables, grouping the data by animal type and outcome type, and then counting the number of occurrences. The result shows clear trends, such as dogs and cats most commonly being transferred or adopted, while other animal types like birds and livestock also show a high frequency of transfers. This insight helps us understand how different animals typically leave the shelter system.

![Query 1](Query1.png)

*(Figure 3. Total number of outcomes by animal type and outcome type)*

**Research Question 2. Which 10 breeds have the highest number of adoption?**

This query identifies the top 10 most adopted breeds and includes the animal type for context. The JOIN statements combine the animals, outcomes, breeds, and outcome_types tables to ensure accurate matching. By filtering for outcome type = 'Adoption', it gives insight into which types and breeds are most frequently adopted. Results show that both cats and dogs dominate the top ranks, with cats slightly more frequent among the top entries.

![Query 2](Query2.png)

*(Figure 4. Top 10 most adopted animal breeds by type)*


**Research Question 3. What is the average time (in days) animals stay in the shelter before an outcome?**

This query calculates the average number of days animals stayed in the shelter before experiencing an outcome (e.g., adoption, euthanasia, etc.). It uses the age_days column from the outcomes table, which was derived from the difference between each animal's date of birth and the date of their outcome.

The result shows that on average, animals stayed in the shelter for approximately 692.01 days, which is useful for evaluating shelter capacity and outcome efficiency over time.

![Query 3](Query3.png)

*(Figure 5. Average number of days animals stayed in the shelter)*


**Research Question 4. How does outcome type vary across age groups?**

This query groups the animals by age categories (based on the `age_days` value) and shows how different outcome types (e.g., Transfer, Adoption, Died) are distributed across those age groups.

It helps identify trends such as:

- Transfers are the most frequent outcome across all age groups.
- Adoptions are most common among younger animals, especially those under 6 months.
- Euthanasia and Died outcomes are more frequent in older age brackets, which may reflect underlying health conditions or shelter policy decisions.
isions.


![Query 4](Query4.png)

*(Figure 6. Outcome type distribution by animal age group)*


**Research Question 5. Which months have the highest number of adoptions?**

This query shows which year-month combinations had the most adoptions, helping to identify seasonal or monthly trends. For example:

- The highest adoptions occurred in June to August 2024, indicating a peak adoption period.
- Certain months in 2023 and earlier also show consistent adoption activity.

This kind of analysis is useful for planning outreach campaigns or understanding when shelters experience the most adoptions.tions.


![Query 5](Query5.png)

*(Figure 7. Months with the highest number of adoptions)*


# Stage 4: Create a Simple Web Application



This stage involves building a simple Node.js web application that connects to the MySQL database and provides insights based on the research questions identified in Stage 1.

The goal of the application is to allow users to query key insights such as adoption trends, outcome types, and average shelter time based on the cleaned and normalized datas.



## Web Application

Using the web application, users can explore different aspects of animal outcomes to gain insights into trends such as adoption rates, outcome types by animal type or age group, and average shelter duration. Each section of the web app corresponds to a specific research question identified earlier in Stage 1.

The purpose of the web application is to make it easier to access and analyze cleaned shelter data. By presenting the results clearly, users can quickly observe patterns such as which breeds are adopted most often or which age groups have higher euthanasia rates.

To run the web application, the user opens a terminal and executes `node app.js`. This launches the Node.js server and connects it to the MySQL database named `animal_outcomes_db`. The server listens on port 3000 and uses the Express framework for routing. The web pages are rendered using EJS templates, while SQL queries are dynamically executed to retrieve data from the database and display it in the browser.

### Accessing the Database

The MySQL connection is handled through the `mysql2` package. Connection details are defined in the `app.js` file. These include the host (`localhost`), user (`root`), password (`12Baseball34`), and database name (`animal_outcomes_db`). Once connected, the application uses SQL queries to fetch data for each research question. The results are passed to EJS templates for rendering.

Each route, such as `/q1` or `/q2`, corresponds to a specific SQL query. When a user visits that route, the application runs the query, receives the data, and displays it in a structured format.

Below is a code snippet from `app.js` showing how the server and database connect, are set up:
s to MySQL, and defines each route:



Below is the key part of the Node.js application (app.js) that sets up the server, connects to MySQL, and defines each route:

![Node.js Setup](Node.js-Setup.png)

*(Figure 8. Node.js server setup with MySQL and Express)*


### Homepage
This is the homepage of the Animal Outcomes Web Application. It provides a clear and simple interface that helps users explore key insights from animal shelter data. From here, users can choose from five research questions, each leading to a separate page where the data is queried and visualized.

The goal is to make the data easy to navigate and understand. Whether someone is interested in finding out which outcomes are most common for different animals, which breeds are adopted the most, or how long animals typically stay in shelters, this homepage serves as the starting point. It also guides users to explore trends by age group and see which months had the highest number of adoptions.

By presenting the data in this way, the homepage allows users to quickly access meaningful patterns and better understand the outcomes recorded in the shelter dataset.t.


![Website Home Page](WebsiteHomePage.png)

*(Figure 9. Animal Outcomes Web Application user interface with research questions)*


**Research Question 1: What are the most common outcome types for each animal type?**

To answer this question, we used a SQL query that joins the `animals`, `outcomes`, and `outcome_types` tables. The data was grouped by `animal_type` and `outcome_type_name`, and the count of each combination was calculated. The goal was to determine which outcome types (such as Transfer, Adoption, etc.) were most frequent for each animal type.

In the earlier version, all results were shown in one long table. This made it harder to read and understand. To improve the user experience, we added a dropdown filter to the web application. Now, users can choose an animal type like Dog, Cat, or Bird and view only the relevant outcomes. This makes it easier to explore specific categories without scrolling through a large dataset.

![Query 1 Code](Query1Code.png)

*(Figure 10. Node.js route and SQL query for Question 1: Outcome types by animal type)*


**Findings:**

From the results, we can see some clear trends:

Dogs were most often recorded with a transfer outcome (1,226 cases), followed by adoption (368). This suggests that dogs are frequently moved between shelters or partner groups before being adopted. There were also 121 cases of return to owner, as well as other outcomes like died, euthanasia, and a few less common categories.

Cats also showed high numbers for transfer (1,161) and adoption (391). While many cats are rehomed successfully, there were also noticeable cases of euthanasia, death, and return to owner, which may reflect health conditions or the challenges of placing certain cats.

Birds had fewer outcomes overall. Most of the birds in the data were recorded as transferred (55), with just a few cases of adoption, euthanasia, or missing. This could mean that birds are usually moved to other facilities rather than adopted directly.

Livestock had very limited data. Only three animals were recorded, all with a transfer outcome. This is likely because livestock are not typically rehomed in the same way as domestic pets.

Other animals, including less common species, were also most frequently transferred (45), but they showed a wider mix of outcomes. These included euthanasia, adoption, and even rare cases of relocate and disposal. This group likely contains animals with more complex care needs or lower adoption demand.

Across all animal types, transfer is the most frequent outcome. This highlights how common it is for shelters to move animals between locations or into partner programs as part of the rehoming process.

![Website Query 1](WebsiteQuery1.png)

*(Figure 11. Web application output for Q1: Most common outcome types by animal type)*


**Research Question 2: What are the top 10 most adopted breeds?**

To explore this question, we used a SQL query that filters for animals with the outcome type labeled as "Adoption". The query brings together data from the animals, outcomes, outcome types, and breeds tables. It counts the number of adoptions for each breed and animal type, then ranks the results to find the 10 breeds with the highest adoption numbers.

This helps us understand which breeds are most commonly adopted, offering insight into public preferences or common trends within shelters.

![Query 2 Code](Query2Code.png)

*(Figure 12. Node.js route and SQL query for Q2: Top 10 adopted breeds)*


**Findings:**

Based on the results, cats appear at the top of the list, with Domestic Shorthair and Domestic Shorthair Mix being the two most adopted breeds overall. This suggests that these cat types are either more commonly available in shelters or particularly preferred by adopters.

When looking at dogs, mixed breeds dominate the rankings. Breeds like Labrador Retriever Mix, Pit Bull Mix, Chihuahua Shorthair Mix, and others made up most of the dog entries in the top 10. These results indicate that people are more likely to adopt mixed breeds, possibly because they’re more frequently surrendered, more affordable, or simply perceived as more approachable.

The table below summarizes the top 10 adopted breeds and their corresponding adoption counts:

![Website Query 2](WebsiteQuery2.png)

*(Figure 13. Web output for Q2: Top 10 adopted breeds)*

**Research Question 3: Average Days in Shelter**

This query calculates the average number of days animals spend in the shelter before an outcome occurs, such as adoption, transfer, or euthanasia. To keep it clear and easy to interpret, we rounded the result to two decimal places.

By knowing this number, we get a clearer idea of how long animals typically remain in shelter care. It also helps highlight potential issues. For example, if animals are staying much longer than expected, it might point to challenges in the rehoming process or in finding suitable adopters.


![Query 3 Code](Query3Code.png)

*(Figure 14. Node.js route and SQL query for Q3: Average days in shelter)*


**Findings:**

The data shows that animals stay in the shelter for an average of 692.01 days before an outcome occurs. While this number might seem high at first, it helps give context to what shelters are dealing with on a daily basis.

A longer shelter stay could be caused by a number of factors. It might mean the shelter is overcrowded, or that some animals are simply harder to adopt because of their breed, age, or medical condition. It could also suggest that adoptions are happening slowly, or that the shelter is taking extra time to ensure animals are placed in the right homes. Either way, this number gives us a useful benchmark.

Understanding this average stay is important for decision-making. If it’s longer than expected, the shelter might want to explore ways to improve adoption rates, create special outreach programs, or prioritize long-term animals. At the same time, a longer stay might reflect the shelter’s commitment to not rushing rehoming decisions and prioritizing animal welfare.

In short, this insight helps the organization better evaluate its processes and make informed changes that could improve both the animals’ experiences and the shelter’s overall performance.


![Website Query 3](WebsiteQuery3.png)

*(Figure 15. Web output for Q3: Average days in shelter)*

**Research Question 4: Outcome Types by Age Group**

This query analyzes the relationship between animal age and the type of outcome they experience at shelters. Animals are divided into four age groups based on the number of days old they are when the outcome occurs:

- Under 6 months  
- 6 months to 3 years  
- 3 to 8 years  
- Over 8 years.

Each group is then analyzed to see which outcomes were most common, such as adoption, transfer, return to owner, euthanasia, or other less frequent events.

The goal is to understand how age affects what happens to animals at the shelter. For example, younger animals might be adopted more quickly, while older animals could face different challenges. This kind of insight can help shelters make more informed decisions about care, outreach, and rehoming strategies.

![Query 4 Code](Query4Code.png)

*(Figure 16. Node.js route and SQL query for Q4: Outcome type by age group)*


The web application displays these results in collapsible sections. Each age group has its own panel that users can expand to see the specific outcome counts. This makes the interface easier to navigate without being overwhelmed by a long list of data.

**Findings:**

Younger animals, especially those under 6 months and those between 6 months and 3 years, showed the highest counts for transfer and adoption outcomes. This suggests that younger animals tend to be rehomed more quickly, possibly because they are considered easier to train, more playful, or more appealing to potential adopters. These age groups also had fewer negative outcomes such as euthanasia or death.

Animals aged 3 to 8 years continued to have strong numbers for adoption and transfer. However, more sensitive outcomes like return to owner, euthanasia, or even death also appeared. This could mean that animals in this age group may face more complex situations, such as longer shelter stays or being surrendered again due to changes in adopter circumstances or age-related concerns.

Older animals, those over 8 years, had noticeably lower adoption and transfer counts. At the same time, they were more likely to experience outcomes such as euthanasia, death, or simply remaining in the shelter for longer. This may be due to age-related health issues, less interest from adopters, or a higher level of care required. It reflects the added challenges shelters face in rehoming senior pets.

By understanding how outcomes vary across age groups, shelters can better plan their support and services. They might introduce adoption drives focused on senior animals, promote fostering for middle-aged pets, or prioritize medical care for long-term residents. At the same time, the data reinforces that younger animals already benefit from strong rehoming trends, which could be extended to others through targeted programs.

![Website Query 4](WebsiteQuery4.png)

*(Figure 17. Outcome types by age group output)*


**Research Question 5: Top Months for Adoptions**

This analysis focuses on identifying the months with the highest number of recorded animal adoptions. Understanding the timing of adoption surges can provide insights into patterns of public interest, the effectiveness of promotional campaigns, or seasonal trends that influence adoption behavior.

To achieve this, the dataset was filtered to include only entries where the outcome type was labeled as "Adoption." The outcome_date field was then formatted by month (%Y-%m) to group the records accordingly. A count of adoptions was performed for each month, and the results were sorted in descending order based on the number of adoptions. Finally, the top 12 months with the highest adoption counts were selected for display.

![Query 5 Code](Query5Code.png)

*(Figure 18. Node.js route and SQL query for Q5: Top months for adoptions)*



**Findings:**

The findings, as presented in the web application (see figure below), reveal that June 2024 had the highest number of adoptions, with a total of 78. This was followed by August 2024 with 70 adoptions, and July 2024 with 67. These three months were the most active, suggesting a possible seasonal trend or the impact of summer campaigns that encouraged more adoptions.

While these summer months dominated the top three, it is also interesting to note that other high-performing months were spread across different years. For example, April and May 2023, as well as April and May 2016, also had notable spikes in adoption activity. This shows that strong adoption numbers are not tied to just one time of year, and may depend on a variety of factors like special events, promotions, or changes in public behavior.

We also saw moderate peaks in months like February 2023 and August 2020, which could be linked to post-holiday rehoming efforts or specific community programs. Even June 2023, while lower than the rest, still made the top 12, showing that some consistency exists in mid-year adoption patterns.

By identifying these high-activity periods, shelters and animal welfare organizations can better plan their outreach and staffing. For example, knowing when adoptions usually increase can help shelters prepare by organizing more volunteers, extending visiting hours, or running targeted campaigns. This helps maximize the chances of animals finding new homes during the months when interest is naturally higher.


![Website Query 5](WebsiteQuery5.png)

*(Figure 19. Top months for adoptions output)*


## Conclusion

This report examined key trends in animal shelter data by exploring five focused questions. Through a combination of data analysis using SQL and improvements in web interface design, it revealed several useful patterns that could benefit shelters, adopters, and decision-makers.

Transfer stood out as the most frequent outcome across all species, especially for dogs and cats. This suggests how vital collaboration between shelters is when finding homes for animals. When looking specifically at adoptions, mixed-breed dogs and Domestic Shorthair cats appeared most often, showing both public interest and what shelters usually have available.

Animals stayed in shelters for an average of 692.01 days. That’s a long time, pointing to either slow adoption processes or extra care being taken with placements. This kind of data helps shelters measure their effectiveness and animal well-being.

Age mattered, too. Younger animals were more likely to be adopted or transferred, while older ones faced higher chances of euthanasia. This shows a need for age-specific strategies to improve outcomes for older pets. Seasonal trends also became clear, adoptions were highest in summer months, hinting that timing and promotion efforts can make a real difference.

In short, this project didn’t just uncover useful findings. It also made things easier for users by adding interactive visuals and better filters. The insights can help shelters make smarter decisions and improve how animals are cared for and rehomed.

## References


### **Dataset**

City of Austin. (n.d.). *Austin Animal Center Outcomes Dataset*. Data.gov. https://data.austintexas.gov/Public-Safety/Austin-Animal-Center-Outcomes/9t4d-g238

### **Libraries and Tools**

The pandas development team. (n.d.). *pandas*

The matplotlib development team. (n.d.). *matplotlib* 

Oracle Corporation. (n.d.). *MySQL* 

Node.js Foundation. (n.d.). *Node.js* 

Express.js Foundation. (n.d.). *Express.js* 

Embedded JavaScript Templates (EJS).

MySQL. (n.d.). *MySQL 8.0 Reference Manual*

Express.js. (n.d.). *Basic routing*

### **Academic Reference**

[1] Silberschatz, A., Korth, H. F., & Sudarshan, S. (2020). Database system concepts (7th ed.). McGraw-Hill Education. https://www.slideshare.net/slideshow/databasesystemconcepts7theditionpdf/266494557

[2] Coronel, C., & Morris, S. (2019). Database systems: Design, implementation, & management (13th ed.). Boston, MA: Cengage Learning. Retrieved from http://121.121.140.173:8887/filesharing/kohasharedfolders/Database%20Systems%20Design,%20Implementation,%20and%20Management%20(Carlos%20Coronel%20Steven%20Morris)%20(2019).pdf

[3] Weiss, E., Miller, K., Mohan-Gibbons, H., & Vela, C. (2012). Why did you choose this pet? Adopters and pet selection preferences in five animal shelters in the United States. Animals, 2(2). https://doi.org/10.3390/ani2020144

[4] Garrison, L., & Weiss, E. (2014). What do people want? Factors people consider when acquiring dogs, the complexity of the choices they make, and implications for nonhuman animal relocation programs. Retrieved from https://www.researchgate.net/publication/264796511

[5] Rahm, E., & Do, H. H. (2000). Data cleaning: Problems and current approaches. IEEE Data Engineering Bulletin. https://www.researchgate.net/publication/220282831_Data_Cleaning_Problems_and_Current_Approaches

[6] Han, J., Pei, J., & Kamber, M. (2011). Data mining: Concepts and techniques (3rd ed.). Morgan Kaufmann. https://www.sku.ac.ir/Datafiles/BookLibrary/43/Data-Mining-Concepts-and-Techniques-Han.pdf

[7] Elmasri, R., & Navathe, S. B. (2015). Fundamentals of Database Systems (7th ed.). Pearson. Available at: https://www.auhd.edu.ye/upfiles/elibrary/Azal2020-01-22-12-28-11-76901.pdf